# S11 · BI/ML distribuido: KPIs + predicción del viento en vivo en Grafana
**Nayder Arce Mayta** · Pushgateway `localhost:49091` · Grafana `localhost:43000` → *Viento Juliaca — Pipeline U2*

1. KPIs batch del flujo (del Parquet de S09) y de la U1 (Gold histórico) → Pushgateway (PUT).
2. Predicción en vivo con el modelo de S10 → Pushgateway por `instance = sensorId`.
3. Auditoría: error real en producción (predicción vs lo que realmente pasó 1 h después).

In [ ]:
import os, time, json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

BASE = os.getenv("VIENTO_BASE", "/opt")
ART = f"{BASE}/artifacts"
KAFKA = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")
TOPIC = "viento-eventos"
PUSHGATEWAY = os.getenv("PUSHGATEWAY", "http://pushgateway:9091")

spark = (SparkSession.builder
         .appName("s11-bi-ml-viento-nayder")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", time.strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
import requests

def push(job, instance, metricas, metodo="PUT"):
    """Publica un grupo de métricas en el Pushgateway (formato texto de Prometheus).
    PUT reemplaza el grupo completo (job, instance)."""
    cuerpo = "".join(f"{k} {float(v)}\n" for k, v in metricas.items() if v is not None)
    url = f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}"
    r = requests.request(metodo, url, data=cuerpo, timeout=5)
    r.raise_for_status()
    return r.status_code

def borrar_grupo(job, instance):
    return requests.delete(f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}", timeout=5).status_code

## 1. KPIs batch del flujo de eventos (volumen, validez, latencia)

In [ ]:
obs = spark.read.parquet(f"{ART}/viento_eventos_observabilidad")
k = obs.agg(F.count("*").alias("total"), F.avg(F.col("isValid").cast("int")).alias("ratio"),
            F.avg("latencyMs").alias("lat"), F.countDistinct("sensorId", "obsTime").alias("lecturas_unicas")).first()
kpis_flujo = {"lambda26_bi_eventos_total": k["total"], "lambda26_bi_eventos_validos_ratio": k["ratio"],
              "lambda26_bi_latencia_promedio_ms": k["lat"], "lambda26_bi_lecturas_unicas": k["lecturas_unicas"]}
print(kpis_flujo, push("lambda26_bi_flujo", "viento_batch", kpis_flujo))

## 2. KPIs de negocio de la capa batch (U1): climatología del mes actual

In [ ]:
gold = spark.read.parquet(f"{ART}/viento_gold")
umbral = json.load(open(f"{ART}/umbral_viento.json"))["umbral_rafaga_kmh"]
mes = int(time.strftime("%m"))
g = gold.filter(F.col("mes") == mes).agg(F.avg("rafaga_max").alias("raf"), F.avg("dia_fuerte").alias("p"),
                                         F.avg("velocidad_media").alias("vel")).first()
kpis_u1 = {"lambda26_bi_u1_rafaga_max_media_mes": g["raf"], "lambda26_bi_u1_prob_dia_fuerte_mes": g["p"],
           "lambda26_bi_u1_velocidad_media_mes": g["vel"], "lambda26_bi_umbral_rafaga_kmh": umbral}
print(kpis_u1, push("lambda26_bi_batch", "u1_gold", kpis_u1))

## 3. Predicción en vivo → Pushgateway (`foreachBatch`)

In [ ]:
# Contrato viento.lectura v1 (docs/contratos.md)
esquema_viento = (StructType()
    .add("tipoEvento", StringType()).add("version", IntegerType())
    .add("sensorId", StringType()).add("estacion", StringType())
    .add("lat", DoubleType()).add("lon", DoubleType())
    .add("velocidadViento", DoubleType()).add("direccionViento", DoubleType()).add("rafaga", DoubleType())
    .add("temperatura", DoubleType()).add("humedad", DoubleType()).add("presion", DoubleType())
    .add("obsTime", LongType()).add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

eventos = (crudo.select("partition", "offset", F.col("timestamp").alias("kafkaTs"),
                        F.col("value").cast("string").alias("json_str"))
           .select("partition", "offset", "kafkaTs", "json_str",
                   F.from_json("json_str", esquema_viento).alias("e"))
           .select("partition", "offset", "kafkaTs", "json_str", "e.*")
           .withColumn("ts", (F.col("timestamp") / 1000).cast("timestamp"))       # tiempo del evento (lectura)
           .withColumn("obs_ts", (F.col("obsTime") / 1000).cast("timestamp")))    # paso de 15 min del modelo
eventos.printSchema()

In [ ]:
from pyspark.ml import PipelineModel
modelo = PipelineModel.load(f"{ART}/modelo_viento_1h")      # de S10, sin reentrenar

def features(df):
    hora = F.hour("ts_feat") + F.minute("ts_feat") / 60.0
    return (df.withColumn("dir_sin", F.sin(F.radians("direccion"))).withColumn("dir_cos", F.cos(F.radians("direccion")))
              .withColumn("hora_sin", F.sin(2 * 3.141592653589793 * hora / 24))
              .withColumn("hora_cos", F.cos(2 * 3.141592653589793 * hora / 24))
              .withColumn("mes_sin", F.sin(2 * 3.141592653589793 * F.month("ts_feat") / 12))
              .withColumn("mes_cos", F.cos(2 * 3.141592653589793 * F.month("ts_feat") / 12)))

P_PRED = f"{ART}/predicciones_stream_s11"
entrada = (eventos.filter((F.col("origen") == "open-meteo") & F.col("velocidadViento").isNotNull())
           .select("sensorId", "estacion", "obs_ts", F.col("velocidadViento").alias("velocidad"), "rafaga",
                   F.col("direccionViento").alias("direccion"), "temperatura", "humedad", "presion"))

def en_vivo(df, batch_id):
    pred = (modelo.transform(features(df.dropDuplicates(["sensorId", "obs_ts"]).withColumn("ts_feat", F.col("obs_ts"))))
            .select(F.lit(batch_id).alias("batchId"), "sensorId", "estacion", F.col("obs_ts").alias("fechaHora"),
                    "velocidad", "rafaga", F.round("prediction", 2).alias("prediccion1h")))
    pred.write.mode("append").parquet(P_PRED)
    for r in pred.orderBy("fechaHora").collect():     # el último push por sensor = la lectura más reciente
        push("lambda26_ml_streaming", r["sensorId"], {
            "lambda26_ml_viento_real": r["velocidad"],
            "lambda26_ml_viento_prediccion_1h": r["prediccion1h"],
            "lambda26_ml_rafaga_real": r["rafaga"],
        })

consulta = (entrada.writeStream.foreachBatch(en_vivo)
            .option("checkpointLocation", f"{ART}/chk_s11_pred").trigger(processingTime="10 seconds").start())

## 4. Confirmar que es en vivo (dos consultas PromQL separadas ≥ 20 s)
En Prometheus (`localhost:49090`) consulta dos veces y captura ambas:
```
lambda26_ml_viento_real{instance="juliaca-viento-01"}
lambda26_ml_viento_prediccion_1h
```
El valor cambia cada vez que Open-Meteo publica un paso nuevo (cada 15 min).

## 5. Auditoría post-hoc: error en producción (requiere ≥ 75 min corriendo)

In [ ]:
consulta.stop()     # detener ANTES de releer
p = spark.read.parquet(P_PRED).dropDuplicates(["sensorId", "fechaHora"])
real = p.select(F.col("sensorId").alias("sid"), F.col("fechaHora").alias("fechaReal"), F.col("velocidad").alias("velocidadReal1h"))
audit = (p.alias("a").join(real.alias("b"), (F.col("a.sensorId") == F.col("b.sid"))
                           & (F.col("b.fechaReal") == F.col("a.fechaHora") + F.expr("INTERVAL 1 HOUR")))
         .select("a.sensorId", "a.fechaHora", "a.velocidad", "a.prediccion1h", "b.velocidadReal1h")
         .withColumn("errorAbsoluto", F.abs(F.col("prediccion1h") - F.col("velocidadReal1h")))
         .withColumn("errorPersistencia", F.abs(F.col("velocidad") - F.col("velocidadReal1h"))))
r = audit.agg(F.count("*").alias("n"), F.avg("errorAbsoluto").alias("mae"), F.avg("errorPersistencia").alias("mae_p")).first()
print(f"totalComparado={r['n']}  maeProduccion={r['mae']}  maePersistenciaProduccion={r['mae_p']}")
audit.orderBy(F.desc("fechaHora")).show(10)
if r["n"]:
    push("lambda26_ml_auditoria", "viento", {"lambda26_ml_mae_produccion": r["mae"],
                                              "lambda26_ml_mae_persistencia_produccion": r["mae_p"],
                                              "lambda26_ml_total_comparado": r["n"]})

## 6. Limpiar el Pushgateway (para que no quede un valor congelado)

In [ ]:
for s in ["juliaca-viento-01", "puno-viento-02", "lampa-viento-03"]:
    print(s, borrar_grupo("lambda26_ml_streaming", s))

**Reflexión (Regla #37 de Google / por qué Pushgateway es un atajo):** el Pushgateway guarda el último
valor empujado y Prometheus lo sigue raspando aunque el job haya muerto → un productor caído se ve
como "valor estable". Para un job continuo lo correcto es exponer `/metrics` propio (pull), o al menos
publicar `push_time_seconds` y alertar si es viejo. Aquí lo usamos porque el notebook es un job efímero.